In [90]:
import pandas as pd
import numpy as np

In [91]:
df=pd.read_csv("Employee_Dataset.csv")
df

,employee_id,department,designation,age,salary,joining_date,last_promotion_date,experience_years,performance_rating,is_active
0,EMP1000,IT,Senior Analyst,60,85000,invalid,01/04/2021,5,3,NaN
1,emp_1,IT,Analyst,NaN,55000,10/06/2020,invalid,5,3,True
2,EMP1002,Sales,NaN,28,85000,NaN,2022-03-01,3,4,NaN
3,EMP1003,it,Analyst,45,35000,invalid,01/04/2021,1,3,NaN
4,NaN,IT,mgr,150,85000,NaN,invalid,12,2,True
...,...,...,...,...,...,...,...,...,...,...
995,EMP1995,Finance,NaN,60,55000,2019-05-10,2022-03-01,1,4,yes
996,EMP1996,IT,Analyst,35,250000,invalid,invalid,-2,5,NaN
997,EMP1997,Sales,Senior Analyst,150,250000,2021/07/15,invalid,3,excellent,yes
998,emp_998,Finance,NaN,60,85000,2019-05-10,NaN,12,NaN,True


1 Convert joining_date to datetime and count how many rows failed conversion.

In [92]:
df['joining_date'].unique()

array(['invalid', '10/06/2020', nan, '2019-05-10', '2021/07/15'],
      dtype=object)

In [93]:
df['joining_date']=pd.to_datetime(df['joining_date'],format="%d-%m-%y",errors="coerce",dayfirst=True)
df['joining_date'].isna().sum()

np.int64(1000)

2 Clean employee_id and identify how many duplicate employees exist after 
standardization. 

In [94]:
df['employee_id'].unique()

array(['EMP1000', 'emp_1', 'EMP1002', 'EMP1003', nan, 'EMP1007',
       'EMP1008', 'EMP1009', 'emp_10', 'emp_12', 'EMP1013', 'emp_15',
       'emp_17', 'emp_18', 'EMP1020', 'emp_22', 'emp_23', 'EMP1024',
       'EMP1027', 'EMP1030', 'emp_32', 'EMP1035', 'EMP1037', 'emp_38',
       'emp_39', 'emp_40', 'EMP1042', 'emp_44', 'emp_52', 'EMP1053',
       'emp_57', 'EMP1058', 'emp_59', 'EMP1063', 'emp_67', 'EMP1069',
       'EMP1070', 'emp_71', 'emp_72', 'EMP1073', 'emp_74', 'EMP1076',
       'EMP1077', 'EMP1078', 'emp_79', 'emp_80', 'EMP1081', 'emp_82',
       'emp_83', 'emp_89', 'emp_91', 'EMP1095', 'emp_96', 'EMP1098',
       'EMP1099', 'EMP1101', 'emp_107', 'emp_109', 'EMP1110', 'EMP1111',
       'emp_115', 'emp_116', 'emp_117', 'EMP1121', 'emp_123', 'emp_125',
       'EMP1126', 'emp_128', 'emp_129', 'EMP1132', 'EMP1135', 'EMP1136',
       'emp_137', 'emp_138', 'emp_139', 'EMP1140', 'EMP1141', 'emp_144',
       'EMP1145', 'emp_147', 'emp_148', 'EMP1149', 'emp_150', 'emp_151',
       'EMP1

In [95]:
df['employee_id']=(
    df['employee_id']
    .str.lower()
    .str.strip()
    .str.replace(r'[^A-Z0-9]','',regex=True)
)
df['employee_id'].duplicated().sum()

np.int64(503)

3 Standardize department and calculate the average salary per department 
excluding invalid salaries.

In [96]:
df['department'].unique()

array(['IT', 'Sales', 'it ', 'HR ', 'sales', 'Finance', nan, 'HR'],
      dtype=object)

In [97]:
df['department']=(df['department']
    .str.title()
    .str.strip()
)
df['department']


0           It
1           It
2        Sales
3           It
4           It
        ...   
995    Finance
996         It
997      Sales
998    Finance
999         It
Name: department, Length: 1000, dtype: object

In [98]:
df['salary']=pd.to_numeric(df['salary'],errors='coerce')
df.groupby('department')['salary'].mean()

department
Finance    113253.012048
Hr         114800.000000
It         107944.444444
Sales      119000.000000
Name: salary, dtype: float64

4. Convert age to numeric and find employees with valid salary but invalid age. 


In [99]:
df['age'].unique()

array(['60', nan, '28', '45', '150', '35', '22', '-5', 'unknown'],
      dtype=object)

In [100]:
df['age']=pd.to_numeric(df['age'],errors='coerce')
df[(df['salary'].notna()) & (df['age'].isna())].shape

(120, 10)

5 .Clean salary and detect outliers using the IQR method.

In [101]:
df['salary'].unique()

array([ 85000.,  55000.,  35000.,     nan, 120000., 250000.])

In [102]:
Q1 = df['salary'].quantile(0.25)
Q3 = df['salary'].quantile(0.75)
IQR = Q3 - Q1

salary_outliers = df[
    (df['salary'] < Q1 - 1.5*IQR) |
    (df['salary'] > Q3 + 1.5*IQR)
]

salary_outliers.shape


(136, 10)

6 Convert performance_rating into numeric and calculate the median rating per 
designation

In [103]:
df['performance'] = pd.to_numeric(df['performance_rating'], errors='coerce')
df['designation'] = df['designation'].str.strip().str.title()

df.groupby('designation')['performance'].median()


designation
Analyst           3.0
Manager           3.0
Mgr               3.0
Senior Analyst    3.0
Name: performance, dtype: float64

7 Identify employees whose last_promotion_date is earlier than their 
joining_date. 

In [104]:
df['last_promotion_date'].unique()

array(['01/04/2021', 'invalid', '2022-03-01', nan], dtype=object)

In [105]:
df['last_promotion_date'] = pd.to_datetime(df['last_promotion_date'], errors='coerce')

df[df['last_promotion_date'] < df['joining_date']].shape


(0, 11)

8 Clean experience_years and find mismatches where experience exceeds 
employee age. 

In [106]:
df['experience_years'] = pd.to_numeric(df['experience_years'], errors='coerce')

df[df['experience_years'] > df['age']].shape


(106, 11)

9. Standardize designation and count how many active employees are in each 
designation. 

In [107]:
df['is_active'] = (
    df['is_active'].astype(str).str.lower()
    .map({'true': True, 'false': False})
)

df[df['is_active'] == True]['designation'].value_counts()


designation
Analyst           64
Mgr               43
Manager           34
Senior Analyst    30
Name: count, dtype: int64

10. Convert is_active to boolean and find inactive employees with recent 
promotions. 

In [108]:
df['last_promotion_date']

0     2021-01-04
1            NaT
2            NaT
3     2021-01-04
4            NaT
         ...    
995          NaT
996          NaT
997          NaT
998          NaT
999          NaT
Name: last_promotion_date, Length: 1000, dtype: datetime64[ns]

In [109]:
df[(df['is_active'] == False) & 
   (df['last_promotion_date'] >= '2022-01-01')].shape


(0, 11)

11. Calculate employee tenure in years and find those with tenure above the 90th 
percentile. 

In [110]:
df['joining_date']

0     NaT
1     NaT
2     NaT
3     NaT
4     NaT
       ..
995   NaT
996   NaT
997   NaT
998   NaT
999   NaT
Name: joining_date, Length: 1000, dtype: datetime64[ns]

In [111]:
df['joining_date'] = (
    (pd.Timestamp.today() - df['joining_date']).dt.days / 365
)

df[df['joining_date'] > df['joining_date'].quantile(0.9)].shape


(0, 11)

12. Identify departments where more than 25% of salary values are missing or invalid. 


In [112]:
(
    df.groupby('department')['salary']
    .apply(lambda x: x.isna().mean())
    .loc[lambda x: x > 0.25]
)


department
Finance    0.366412
Hr         0.321267
It         0.340659
Sales      0.398340
Name: salary, dtype: float64

13. Create a flag for employees with high performance (≥4) but below-median salary. 

In [113]:
median_salary = df['salary'].median()

df['high_perf_low_pay'] = (
    (df['performance'] >= 4) &
    (df['salary'] < median_salary)
)

df['high_perf_low_pay'].sum()


np.int64(58)

14. Detect employees with no promotion date but more than 5 years of experience.

In [114]:
df[(df['last_promotion_date'].isna()) &
   (df['experience_years'] > 5)].shape


(191, 12)

15. Build a validation rule to flag rows violating at least two business constraints.

In [115]:
df['violation_count'] = (
    (df['age'] > 100).astype(int) +
    (df['experience_years'] > df['age']).astype(int) +
    (df['salary'].isna()).astype(int) +
    (df['joining_date'].isna()).astype(int)
)

df['rule_violation'] =_attach = df['violation_count'] >= 2
df['rule_violation'].sum()


np.int64(489)

In [116]:
df['last_promotion_date']
df['performance']

0      3.0
1      3.0
2      4.0
3      3.0
4      2.0
      ... 
995    4.0
996    5.0
997    NaN
998    NaN
999    5.0
Name: performance, Length: 1000, dtype: float64

In [118]:
df['last_promotion_date'] = pd.to_datetime(
    df['last_promotion_date'], errors='coerce'
)

df['joining_date'] = pd.to_datetime(
    df['joining_date'], errors='coerce'
)

df['promotion_gap_years'] = (
    (df['last_promotion_date'] - df['joining_date'])
    .dt.days / 365
)

promo_gap_lt_1yr = df[
    (df['promotion_gap_years'].notna()) &
    (df['promotion_gap_years'] < 1)
]

promo_gap_lt_1yr.shape


(0, 15)

2

In [ ]:
dept_avg_salary = df.groupby('department')['salary'].transform('mean')
dept_median_perf = df.groupby('department')['performance'].transform('median')

salary_perf_mismatch = df[
    (df['salary'] > dept_avg_salary) &
    (df['performance'] < dept_median_perf)
]

salary_perf_mismatch.shape


(51, 11)

3

In [ ]:
def age_band(age):
    if age < 30:
        return 'Young'
    elif age <= 50:
        return 'Mid'
    else:
        return 'Senior'

df['age_band'] = df['age'].apply(age_band)

age_band_dept_count = (
    df.groupby(['department', 'age'])
    .size()
    .reset_index(name='count')
)

age_band_dept_count


,department,age,count
0,Finance,-5.0,22
1,Finance,22.0,18
2,Finance,28.0,15
3,Finance,35.0,16
4,Finance,45.0,15
5,Finance,60.0,13
6,Finance,150.0,11
7,Hr,-5.0,26
8,Hr,22.0,20
9,Hr,28.0,34


4

In [ ]:
company_avg_exp = df['experience_years'].mean()

dept_high_exp = (
    df.groupby('department')['experience_years']
    .mean()
    .loc[lambda x: x > company_avg_exp]
)

dept_high_exp


department
Finance    4.826923
Sales      4.709677
Name: experience_years, dtype: float64

5

In [ ]:
recent_cutoff = pd.Timestamp.today() - pd.DateOffset(years=2)

inactive_recent_promo = df[
    (df['is_active'] == False) &
    (df['last_promotion_date'] >= recent_cutoff)
]

inactive_recent_promo.shape


(0, 12)

6

In [ ]:
designation_exp_issue = (
    df.groupby('designation')['experience_years']
    .apply(lambda x: x.isna().mean())
)

designation_exp_issue[designation_exp_issue > 0.20]


designation
Mgr               0.264045
Senior Analyst    0.202128
Name: experience_years, dtype: float64

7

In [ ]:
df['salary_exp_ratio'] = df['salary'] / df['experience_years']

ratio_95 = df['salary_exp_ratio'].quantile(0.95)

extreme_salary_exp = df[df['salary_exp_ratio'] > ratio_95]
extreme_salary_exp.shape


(16, 13)

8

In [ ]:
age_exp_inconsistent = df[
    (df['age'].notna()) &
    (df['experience_years'] > df['age'] - 18)
]

age_exp_inconsistent.shape


(156, 13)

9

In [ ]:
attrition_rate = (
    df.groupby('department')['is_active']
    .apply(lambda x: (x == False).mean())
    .reset_index(name='attrition_rate')
)

attrition_rate


,department,attrition_rate
0,Finance,0.160305
1,Hr,0.208145
2,It,0.201465
3,Sales,0.195021


10

In [ ]:
df['valid_field_count'] = (
    df['joining_date'].notna().astype(int) +
    df['salary'].notna().astype(int) +
    df['age'].notna().astype(int) +
    df['experience_years'].notna().astype(int) +
    df['performance'].notna().astype(int) +
    df['department'].notna().astype(int) +
    df['designation'].notna().astype(int)
)
